In [1]:
import pandas as pd
import json
from pathlib import Path
import re
from difflib import SequenceMatcher


In [2]:
import sys
print(sys.executable)

/home/datalab/projects/YpD-take-home-exercise/.venv_ypd/bin/python


In [ ]:
DATA_DIR = Path("/home/datalab/projects/YpD-take-home-exercise")

TECH_NEWS_FILE = DATA_DIR / "tech_news.csv"
COMPANY_METADATA_FILE = DATA_DIR / "company_metadata.json"

In [11]:
# https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html
articles_df = pd.read_csv(TECH_NEWS_FILE,dtype=str)

articles_df.head()

,article_id,title,company_name,published_date,category,revenue,summary,url,author,word_count
0,ART0001,Scale AI Raises Series D Funding Round,Scale AI,21 Feb 2020,Financial Technology,NaN,The company demonstrated significant advances ...,https://technews.example.com/articles/1,John Smith,2569.0
1,ART0002,DataRobot Announces Breakthrough in Large Lang...,DataRobot,02/23/2023,Software,"£244,094,488",Strategic acquisition strengthens competitive ...,https://technews.example.com/articles/2,Alex Johnson,1878.0
2,ART0003,Meta AI Raises Series D Funding Round,Meta AI,2022-02-17,Analytics,NaN,Leadership outlines vision for responsible AI ...,https://technews.example.com/articles/3,John Smith,1981.0
3,ART0004,Confluent Reports Record Revenue Growth,Confluent,2023-12-13,Artificial Intelligence,$980.0M,Investors show strong confidence in the compan...,https://technews.example.com/articles/4,Sam Wilson,1994.0
4,ART0005,Amazon Web Services Acquires Competitor for Un...,Amazon Web Services,12/28/2021,Financial Technology,Not disclosed,Leadership outlines vision for responsible AI ...,https://technews.example.com/articles/5,Alex Johnson,2781.0


In [12]:
# https://docs.python.org/uk/3/library/json.html
with open(COMPANY_METADATA_FILE, "r", encoding="utf-8") as file:
    company_metadata = json.load(file)

In [13]:
# https://pandas.pydata.org/pandas-docs/version/2.3/reference/api/pandas.DataFrame.from_dict.html
companies_df = (
    pd.DataFrame.from_dict(company_metadata, orient="index", dtype=str)
    .reset_index()
    .rename(columns={"index": "company_name"})
)

companies_df.head()

,company_name,founded_year,headquarters,employee_count,industry,is_public,stock_ticker
0,OpenAI,2006,"Austin, TX",12793,Data Analytics,False,OPEN
1,Anthropic,2006,"San Francisco, CA",43747,FinTech,False,NaN
2,Google DeepMind,2017,"Austin, TX",24274,Data Analytics,False,NaN
3,Meta AI,2001,"Seattle, WA",20240,Data Analytics,True,NaN
4,Microsoft,2018,"Austin, TX",48646,Data Analytics,False,NaN


In [14]:
import re
import math
import pandas as pd
from datetime import datetime


EUR_TO_USD = 1.10
GBP_TO_USD = 1.27
JPY_TO_USD = 1 / 150

MISSING_VALUES = {
    "n/a",
    "na",
    "nan",
    "none",
    "null",
    "not disclosed",
    "undisclosed",
    "missing",
    "",
    "not available",
}

def parse_arr_usd(value):
    """
    Parse a messy ARR/revenue string and return its value in USD as an integer.

    Returns None for missing, invalid, or undisclosed values.
    """

    # ---------------------------------------------------------
    # 1. Handle null / missing values
    # ---------------------------------------------------------
    if value is None:
        return None

    text = str(value).strip()

    if not text:
        return None


    if text.lower() in MISSING_VALUES:
        return None

    # ---------------------------------------------------------
    # 2. Detect currency
    # ---------------------------------------------------------
    text_upper = text.upper()

    if "€" in text or "EUR" in text_upper:
        conversion_rate = EUR_TO_USD

    elif "£" in text or "GBP" in text_upper:
        conversion_rate = GBP_TO_USD

    elif "¥" in text or "JPY" in text_upper:
        conversion_rate = JPY_TO_USD

    else:
        conversion_rate = 1.0

    # ---------------------------------------------------------
    # 3. Remove currency identifiers
    # ---------------------------------------------------------
    cleaned = text_upper

    for token in ["USD", "EUR", "GBP", "JPY", "$", "€", "£", "¥"]:
        cleaned = cleaned.replace(token, "")

    cleaned = cleaned.strip()

    # ---------------------------------------------------------
    # 4. Parse ranges
    # ---------------------------------------------------------
    range_parts = re.split(r"\s*[-–—]\s*", cleaned)

    if len(range_parts) == 2:
        first_value = _parse_numeric_amount(range_parts[0])
        second_value = _parse_numeric_amount(range_parts[1])

        if first_value is None or second_value is None:
            return None

        midpoint = (first_value + second_value) / 2
        usd_value = midpoint * conversion_rate

    else:
        # -----------------------------------------------------
        # 5. Parse single amount
        # -----------------------------------------------------
        parsed_value = _parse_numeric_amount(cleaned)

        if parsed_value is None:
            return None

        usd_value = parsed_value * conversion_rate

    # ---------------------------------------------------------
    # 6. Protect against NaN / inf
    # ---------------------------------------------------------
    if not math.isfinite(usd_value):
        return None

    return int(round(usd_value))


def _parse_numeric_amount(text):
    """
    Parse values like:
        5.2B
        500M
        5.2 BILLION
        5,200,000,000
    """

    if text is None:
        return None

    text = str(text).strip().upper()

    if not text or text in {"NAN", "NULL", "NONE", "N/A", "NA"}:
        return None

    text = text.replace(",", "")

    multiplier = 1

    if re.search(r"\bBILLION\b", text):
        multiplier = 1_000_000_000
        text = re.sub(r"\bBILLION\b", "", text)

    elif re.search(r"\bMILLION\b", text):
        multiplier = 1_000_000
        text = re.sub(r"\bMILLION\b", "", text)

    elif text.endswith("B"):
        multiplier = 1_000_000_000
        text = text[:-1]

    elif text.endswith("M"):
        multiplier = 1_000_000
        text = text[:-1]

    text = text.strip()

    try:
        numeric_value = float(text)
    except ValueError:
        return None

    if not math.isfinite(numeric_value):
        return None

    return numeric_value * multiplier

In [27]:
def parse_published_date(value, ambiguous_day_first=False):
    """
    Parse a messy published_date value into a UTC-aware pandas Timestamp.

    Parameters
    ----------
    value : Any
        Raw date value from the source dataset.

    ambiguous_day_first : bool, default False
        Controls how ambiguous numeric dates are interpreted.

        False:
            06/09/2024 -> June 9, 2024

        True:
            06/09/2024 -> September 6, 2024

    Returns
    -------
    pandas.Timestamp | pd.NaT
        Parsed UTC-aware timestamp or pd.NaT when invalid/missing.
    """

    # ---------------------------------------------------------
    # 1. Handle missing values
    # ---------------------------------------------------------
    if value is None:
        return pd.NaT

    text = str(value).strip()

    if text.lower() in MISSING_VALUES:
        return pd.NaT

    # ---------------------------------------------------------
    # 2. Try known explicit formats first
    # ---------------------------------------------------------
    known_formats = [
        "%Y-%m-%d",      # 2024-09-06
        "%Y/%m/%d",      # 2024/09/06
        "%Y.%m.%d",      # 2024.09.06

        "%B %d, %Y",     # September 6, 2024
        "%b %d, %Y",     # Sep 6, 2024

        "%d %B %Y",      # 6 September 2024
        "%d %b %Y",      # 6 Sep 2024
    ]

    for fmt in known_formats:
        try:
            parsed_date = datetime.strptime(text, fmt)

            # Explicit formats do not contain timezone information,
            # so assume UTC for consistency across the dataset.
            return pd.Timestamp(parsed_date, tz="UTC")

        except ValueError:
            pass

    # ---------------------------------------------------------
    # 3. Fallback for numeric / mixed formats
    # ---------------------------------------------------------
    try:
        parsed_date = pd.to_datetime(
            text,
            errors="raise",
            dayfirst=ambiguous_day_first,
            utc=True
        )

        return parsed_date

    except (ValueError, TypeError):
        return pd.NaT

In [15]:
CATEGORY_MAPPING = {
    "ai/ml": "AI_ML",
    "ai & ml": "AI_ML",
    "ai and ml": "AI_ML",
    "artificial intelligence": "AI_ML",
    "machine learning": "AI_ML",
    "ai": "AI_ML",
    "ml": "AI_ML",

    "cloud": "CLOUD",
    "cloud computing": "CLOUD",

    "data analytics": "DATA_ANALYTICS",
    "analytics": "DATA_ANALYTICS",

    "cybersecurity": "CYBERSECURITY",
    "cyber security": "CYBERSECURITY",

    "fintech": "FINTECH",
    "financial technology": "FINTECH",

    "saas": "SAAS",
    "software as a service": "SAAS",
}


def normalize_category(value):
    """
    Normalize messy article category values into a consistent taxonomy.

    Returns
    -------
    str | None
        Standardized category value, or None for missing values.
    """

    # ---------------------------------------------------------
    # 1. Handle missing values
    # ---------------------------------------------------------
    if value is None:
        return None

    text = str(value).strip()

    if text.lower() in MISSING_VALUES:
        return None

    # ---------------------------------------------------------
    # 2. Normalize formatting
    # ---------------------------------------------------------
    normalized_text = text.lower()

    normalized_text = normalized_text.replace("&", "and")

    # Collapse multiple spaces
    normalized_text = re.sub(r"\s+", " ", normalized_text)

    # Remove leading/trailing spaces
    normalized_text = normalized_text.strip()

    # ---------------------------------------------------------
    # 3. Apply explicit mapping
    # ---------------------------------------------------------
    if normalized_text in CATEGORY_MAPPING:
        return CATEGORY_MAPPING[normalized_text]

    # ---------------------------------------------------------
    # 4. Keep unknown categories instead of silently dropping
    # ---------------------------------------------------------
    return normalized_text.upper().replace(" ", "_")

In [16]:
COMPANY_ALIASES = {
    "aws": "Amazon Web Services",
    "amazon web services aws": "Amazon Web Services",

    "open ai": "OpenAI",
    "openai inc": "OpenAI",

    "nvidia corporation": "NVIDIA",
    "nvidia corp": "NVIDIA",

    "databricks inc": "Databricks",

    "snowflake inc": "Snowflake",

    "mongodb inc": "MongoDB",

    "meta ai research": "Meta AI",
}


def _normalize_company_name(value):
    """
    Normalize a company name for comparison only.

    This does NOT become the final company name.
    """

    if value is None:
        return None

    text = str(value).strip().lower()

    if text in MISSING_VALUES:
        return None

    # Replace "&" consistently
    text = text.replace("&", "and")

    # Remove punctuation
    text = re.sub(r"[^\w\s]", " ", text)

    # Collapse multiple spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [17]:
def resolve_company_name(
    value,
    canonical_companies,
    fuzzy_threshold=0.90
):
    """
    Resolve a source company name to a canonical company name.

    Matching order:
        1. Exact normalized match
        2. Explicit alias mapping
        3. Conservative fuzzy matching

    Parameters
    ----------
    value : str
        Raw company name from the article dataset.

    canonical_companies : iterable
        Canonical company names from the metadata dataset.

    fuzzy_threshold : float, default 0.90
        Minimum similarity required for fuzzy matching.

    Returns
    -------
    str | None
        Canonical company name, or None if no reliable match exists.
    """

    normalized_value = _normalize_company_name(value)

    if normalized_value is None:
        return None

    # ---------------------------------------------------------
    # 1. Build normalized lookup from metadata
    # ---------------------------------------------------------
    canonical_lookup = {
        _normalize_company_name(company): company
        for company in canonical_companies
    }

    # ---------------------------------------------------------
    # 2. Exact normalized match
    # ---------------------------------------------------------
    if normalized_value in canonical_lookup:
        return canonical_lookup[normalized_value]

    # ---------------------------------------------------------
    # 3. Explicit aliases
    # ---------------------------------------------------------
    if normalized_value in COMPANY_ALIASES:
        alias_target = COMPANY_ALIASES[normalized_value]

        # Validate that alias target actually exists in metadata
        if alias_target in canonical_companies:
            return alias_target

    # ---------------------------------------------------------
    # 4. Fuzzy match
    # ---------------------------------------------------------
    best_match = None
    best_score = 0

    for normalized_canonical, canonical_name in canonical_lookup.items():

        score = SequenceMatcher(
            None,
            normalized_value,
            normalized_canonical
        ).ratio()

        if score > best_score:
            best_score = score
            best_match = canonical_name

    if best_score >= fuzzy_threshold:
        return best_match

    # ---------------------------------------------------------
    # 5. Unresolved
    # ---------------------------------------------------------
    return None

In [18]:
def derive_company_size(employee_count):
    """
    Derive company size category from employee count.

    Rules:
        Small  -> fewer than 10,000 employees
        Medium -> 10,000 through 30,000 employees
        Large  -> more than 30,000 employees

    Returns
    -------
    str | None
        SMALL, MEDIUM, LARGE, or None for missing/invalid values.
    """

    if employee_count is None:
        return None

    text = str(employee_count).strip()

    if text.lower() in MISSING_VALUES:
        return None

    try:
        count = int(float(text))
    except (ValueError, TypeError):
        return None

    if count < 0:
        return None

    if count < 10_000:
        return "SMALL"

    elif count <= 30_000:
        return "MEDIUM"

    else:
        return "LARGE"

In [19]:
def derive_company_age(founded_year, published_date):
    """
    Calculate company age as of the article published date.

    Parameters
    ----------
    founded_year : int | str
        Company founding year.

    published_date : pandas.Timestamp | datetime | str
        Cleaned article publication date.

    Returns
    -------
    int | None
        Company age in years, or None if inputs are missing/invalid.
    """

    # Handle missing founded year
    if founded_year is None:
        return None

    founded_text = str(founded_year).strip().lower()

    if founded_text in MISSING_VALUES:
        return None

    try:
        founded_year = int(float(founded_year))
    except (ValueError, TypeError):
        return None

    # Handle missing / invalid published date
    if published_date is None or pd.isna(published_date):
        return None

    try:
        published_date = pd.Timestamp(published_date)
    except (ValueError, TypeError):
        return None

    company_age = published_date.year - founded_year

    # Prevent logically invalid negative ages
    if company_age < 0:
        return None

    return company_age

In [20]:
canonical_companies = companies_df["company_name"].tolist()

dim_company = companies_df.copy(deep=True)

dim_company["company_size_category"] = (
    dim_company["employee_count"]
    .apply(derive_company_size)
)

In [22]:
articles_df

,article_id,title,company_name,published_date,category,revenue,summary,url,author,word_count
0,ART0001,Scale AI Raises Series D Funding Round,Scale AI,21 Feb 2020,Financial Technology,NaN,The company demonstrated significant advances ...,https://technews.example.com/articles/1,John Smith,2569.0
1,ART0002,DataRobot Announces Breakthrough in Large Lang...,DataRobot,02/23/2023,Software,"£244,094,488",Strategic acquisition strengthens competitive ...,https://technews.example.com/articles/2,Alex Johnson,1878.0
2,ART0003,Meta AI Raises Series D Funding Round,Meta AI,2022-02-17,Analytics,NaN,Leadership outlines vision for responsible AI ...,https://technews.example.com/articles/3,John Smith,1981.0
3,ART0004,Confluent Reports Record Revenue Growth,Confluent,2023-12-13,Artificial Intelligence,$980.0M,Investors show strong confidence in the compan...,https://technews.example.com/articles/4,Sam Wilson,1994.0
4,ART0005,Amazon Web Services Acquires Competitor for Un...,Amazon Web Services,12/28/2021,Financial Technology,Not disclosed,Leadership outlines vision for responsible AI ...,https://technews.example.com/articles/5,Alex Johnson,2781.0
...,...,...,...,...,...,...,...,...,...,...
745,ART0746,Stripe Inc. Acquires Competitor for Undisclose...,Stripe Inc.,"December 02, 2024",SaaS,"£3,078,740,157",New product features leverage cutting-edge AI ...,https://technews.example.com/articles/746,John Smith,2891.0
746,ART0747,Confluent Faces Regulatory Scrutiny,Confluent,2022-01-26T00:00:00Z,Cloud,"¥90,000,000,000",Investors show strong confidence in the compan...,https://technews.example.com/articles/747,John Smith,1990.0
747,ART0748,MongoDB Partners with Major Enterprise Client,MongoDB,11-13-2024,Analytics,$1.954B - $2.160B,Company reaches important financial milestone ...,https://technews.example.com/articles/748,Jane Doe,2175.0
748,ART0749,xAI Faces Regulatory Scrutiny,xAI,2024-11-07,Machine Learning,$180.0 million,International expansion strategy focuses on en...,https://technews.example.com/articles/749,Jane Doe,2458.0


In [26]:
fact_article_arr = articles_df.copy(deep=True)
fact_article_arr["published_date"]

0               21 Feb 2020
1                02/23/2023
2                2022-02-17
3                2023-12-13
4                12/28/2021
               ...         
745       December 02, 2024
746    2022-01-26T00:00:00Z
747              11-13-2024
748              2024-11-07
749              07/15/2024
Name: published_date, Length: 750, dtype: str

In [28]:
fact_article_arr = articles_df.copy(deep=True)

fact_article_arr["arr_usd"] = fact_article_arr["revenue"].apply(parse_arr_usd)
fact_article_arr["published_date_clean"] = fact_article_arr["published_date"].apply(parse_published_date)
fact_article_arr["category_clean"] = articles_df["category"].apply(normalize_category)

fact_article_arr["company_name_resolved"] = fact_article_arr["company_name"].apply(
        lambda x: resolve_company_name(
            x,
            canonical_companies
        )
)


/tmp/ipykernel_917924/3421927585.py:66: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  parsed_date = pd.to_datetime(


In [29]:
fact_article_arr_enriched = fact_article_arr.merge(
    dim_company,
    left_on="company_name_resolved",
    right_on="company_name",
    how="left",
    suffixes=("", "_y")
).drop("company_name_y",
  axis="columns")

In [30]:
fact_article_arr_enriched["company_age"] = fact_article_arr_enriched.apply(
    lambda row: derive_company_age(
        row["founded_year"],
        row["published_date_clean"]
    ),
    axis=1
)

In [31]:
fact_article_arr_enriched.columns

Index(['article_id', 'title', 'company_name', 'published_date', 'category',
       'revenue', 'summary', 'url', 'author', 'word_count', 'arr_usd',
       'published_date_clean', 'category_clean', 'company_name_resolved',
       'founded_year', 'headquarters', 'employee_count', 'industry',
       'is_public', 'stock_ticker', 'company_size_category', 'company_age'],
      dtype='str')

In [32]:
duplicate_counts = (
    fact_article_arr_enriched["article_id"]
    .value_counts()
    .loc[lambda x: x > 1]
)

duplicates_df = fact_article_arr_enriched[
    fact_article_arr_enriched["article_id"].isin(duplicate_counts.index)
].sort_values("article_id")
duplicates_df

,article_id,title,company_name,published_date,category,revenue,summary,url,author,word_count,...,category_clean,company_name_resolved,founded_year,headquarters,employee_count,industry,is_public,stock_ticker,company_size_category,company_age


In [35]:
ai_articles_enriched

,article_id,title,company_name,published_date,category,arr_usd,summary,url,industry,founded_year,headquarters,employee_count,is_public,stock_ticker,company_age,company_size_category
3,ART0004,Confluent Reports Record Revenue Growth,Confluent,2023-12-13 00:00:00+00:00,AI_ML,9.800000e+08,Investors show strong confidence in the compan...,https://technews.example.com/articles/4,Cloud Computing,1995,"London, UK",3884,False,NaN,28.0,SMALL
8,ART0009,MongoDB Expands into European Market,MongoDB,2023-02-07 00:00:00+00:00,AI_ML,1.480000e+09,Leadership outlines vision for responsible AI ...,https://technews.example.com/articles/9,SaaS,2008,"Austin, TX",16952,False,MONG,15.0,MEDIUM
28,ART0029,NVIDIA Announces Breakthrough in Large Languag...,NVIDIA,2023-09-13 00:00:00+00:00,BIG_DATA,4.050000e+10,Company reaches important financial milestone ...,https://technews.example.com/articles/29,AI/ML,2017,"London, UK",22031,False,NaN,6.0,MEDIUM
32,ART0033,SpaceX Achieves Profitability Milestone,SpaceX,2022-02-15 00:00:00+00:00,CLOUD,5.000000e+09,International expansion strategy focuses on en...,https://technews.example.com/articles/33,AI/ML,2016,"Berlin, Germany",31018,False,NaN,6.0,LARGE
35,ART0036,NVIDIA Launches New AI-Powered Product,NVIDIA,2023-09-05 00:00:00+00:00,AI_ML,4.050000e+10,Major partnership validates technology and ope...,https://technews.example.com/articles/36,AI/ML,2017,"London, UK",22031,False,NaN,6.0,MEDIUM
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
739,ART0740,Tesla Achieves Profitability Milestone,Tesla,2024-07-01 00:00:00+00:00,AI_ML,9.888000e+10,Platform improvements deliver enhanced perform...,https://technews.example.com/articles/740,Cloud Computing,2008,"London, UK",5691,True,NaN,16.0,SMALL
740,ART0741,Scale AI CEO Discusses Future of AI,Scale AI,2024-01-05 00:00:00+00:00,AI_ML,8.970000e+08,International expansion strategy focuses on en...,https://technews.example.com/articles/741,AI/ML,2003,"Berlin, Germany",23379,False,NaN,21.0,MEDIUM
742,ART0743,SpaceX Expands into European Market,SpaceX,2024-10-22 00:00:00+00:00,FINTECH,9.945000e+09,New product features leverage cutting-edge AI ...,https://technews.example.com/articles/743,AI/ML,2016,"Berlin, Germany",31018,False,NaN,8.0,LARGE
744,ART0745,NVIDIA Unveils Next-Generation Platform,NVIDIA,2024-09-08 00:00:00+00:00,FINTECH,5.850000e+10,The company demonstrated significant advances ...,https://technews.example.com/articles/745,AI/ML,2017,"London, UK",22031,False,NaN,7.0,MEDIUM


In [4]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [40]:
fact_article_arr_enriched["embedding_text"] = (
    fact_article_arr_enriched["title"].fillna("")
    + " "
    + fact_article_arr_enriched["summary"].fillna("")
).str.strip()

In [41]:
embeddings = embedding_model.encode(
    fact_article_arr_enriched["embedding_text"].tolist(),
    show_progress_bar=True
)

Batches:   0%|          | 0/24 [00:00<?, ?it/s]

In [42]:
fact_article_arr_enriched["embedding"] = list(embeddings)

In [43]:
fact_article_arr_enriched["embedding"]

0      [-0.028980197, -0.035523895, -0.0061523053, 0....
1      [-0.0123871155, -0.11811704, 0.009584922, -0.0...
2      [-0.0049830433, -0.0396686, 0.03469421, -0.020...
3      [-0.01743807, -0.09429833, -0.025430625, 0.008...
4      [-0.08201267, -0.03712081, -0.023182329, -0.02...
                             ...                        
745    [-0.15068498, 0.007389304, -0.031167801, -0.04...
746    [0.02430266, -0.091632225, -0.0132649485, 0.04...
747    [-0.025004866, 0.070931286, -0.043854434, 0.03...
748    [0.06897676, -0.06313853, -0.021407384, -0.052...
749    [-0.017552279, 0.013664385, -0.06341434, -0.04...
Name: embedding, Length: 750, dtype: object

In [47]:
import json

fact_article_arr_enriched["embedding_serialized"] = (
    fact_article_arr_enriched["embedding"]
    .apply(lambda x: json.dumps(x.tolist()))
)

In [49]:
ai_articles_enriched = (
    fact_article_arr_enriched[
        (
            (fact_article_arr_enriched["category_clean"] == "AI_ML")
            | (fact_article_arr_enriched["industry"] == "AI/ML")
        )
        & (
            fact_article_arr_enriched["published_date_clean"]
            >= pd.Timestamp("2022-01-01", tz="UTC")
        )
        & (
            fact_article_arr_enriched["published_date_clean"]
            < pd.Timestamp("2025-01-01", tz="UTC")
        )
        & (
            fact_article_arr_enriched["arr_usd"].notna()
        )
        & (
            fact_article_arr_enriched["arr_usd"] > 50_000_000
        )
    ]
    .copy(deep=True)
)

ai_articles_enriched = (
    ai_articles_enriched[
        [
            "article_id",
            "title",
            "company_name_resolved",
            "published_date_clean",
            "category_clean",
            "arr_usd",
            "summary",
            "url",
            "industry",
            "founded_year",
            "headquarters",
            "employee_count",
            "is_public",
            "stock_ticker",
            "company_age",
            "company_size_category",
            "embedding_serialized"
        ]
    ]
    .rename(
        columns={
            "company_name_resolved": "company_name",
            "published_date_clean": "published_date",
            "category_clean": "category",
            "embedding_serialized" : "embedding"
        }
    )
)

dim_company.to_csv(
    "dim_company.csv",
    index=False
)

fact_article_arr.to_csv(
    "fact_article_arr.csv",
    index=False
)

fact_article_arr_enriched.to_csv(
    "fact_article_arr_enriched.csv",
    index=False
)

ai_articles_enriched.to_csv(
    "ai_articles_enriched.csv",
    index=False
)



In [ ]:
assert (
    ai_articles_enriched["arr_usd"] > 50_000_000
).all()

assert (
    ai_articles_enriched["published_date"].dt.year
    .between(2022, 2024)
).all()

assert (
    (ai_articles_enriched["category"] == "AI_ML")
    | (ai_articles_enriched["industry"] == "AI/ML")
).all()


In [50]:
import numpy as np


def cosine_similarity(vector_a, vector_b):
    """
    Calculate cosine similarity between two vectors.

    Returns
    -------
    float
        Similarity score between -1 and 1.
        Higher values indicate greater semantic similarity.
    """

    vector_a = np.asarray(vector_a)
    vector_b = np.asarray(vector_b)

    denominator = (
        np.linalg.norm(vector_a)
        * np.linalg.norm(vector_b)
    )

    if denominator == 0:
        return 0.0

    return np.dot(vector_a, vector_b) / denominator

In [ ]:
import duckdb


IOException: IO Error: Could not set lock on file "/home/datalab/projects/YpD-take-home-exercise/tech_news.duckdb": Conflicting lock is held in /home/datalab/miniconda3/bin/python3.12 (PID 1023064) by user datalab. See also https://duckdb.org/docs/stable/connect/concurrency

In [5]:
con = duckdb.connect("tech_news.duckdb")
duckdb_articles_df = fact_article_arr_enriched.copy()

duckdb_articles_df["embedding"] = (
    duckdb_articles_df["embedding"]
    .apply(
        lambda x: [float(value) for value in x]
    )
)

NameError: name 'fact_article_arr_enriched' is not defined